# Node 2: Agentic Claims Investigator (Generative AI Clinical Reasoner)

**Hackathon Proof of Concept**
**Component 2 of 2: the reasoning layer behind the Cotiviti Agentic Audit Copilot**

The first notebook (`Claims_Anomaly_Detector.ipynb`) implements **Node 1**:
a statistical trigger engine. Rolling z-scores flag claims that deviate
from a provider's normal billing pattern — the kind of signal that surfaces
pre-pay DRG upcoding or operational spikes. Node 1 answers
**"is this unusual?"** but not **"why, and what should an investigator do
about it?"**

This notebook is **Node 2**: a real model, called through the
LLM (can use any but used Anthropic) API, that takes each claim Node 1 flagged and investigates it —
gathering simulated supporting evidence, reasoning about it step by step,
and producing a structured, human-reviewable recommendation. Together, the
two notebooks form the Human-in-the-Loop (HITL) Adjudication Layer
described in the written report: Node 1 decides *which* claims are worth
investigating, and Node 2 investigates *why*, narrates the reasoning, and
hands a human investigator an audit-ready recommendation — never an
autonomous payment verdict.

This is a genuine multi-step LLM pipeline, every
reasoning step below is an actual API call to LLM, and the model's
wording will vary slightly each time you run it.

**Why scripted steps rather than autonomous tool-calling:** a fixed
3-step chain (triage → evidence review → recommendation) keeps the demo
fast, cheap, and easy to read end to end, while still showing genuine
multi-step LLM reasoning chained together — exactly the "chain reasoning"
behavior called out in the assessment topic. A production system would
likely give LLM real tool-calling to decide *which* evidence to pull;
here the evidence is simulated and handed to the model directly so the
demo runs in seconds without needing a real claims database.

**Important:** this is a decision-support assistant, not an autonomous
approver. Every output below ends in a recommendation for a human
investigator — never an automatic payment action. This mirrors the report's
explicit design principle: *"Rather than issuing autonomous payment
verdicts, this system flags aberrant utilization behavior... The model
ranks records by financial risk and generates an auditable, narrative
summary explaining why each item was surfaced."*


## 1. Setup and API key check

This notebook makes real calls to the Claude API and therefore needs an
Anthropic API key. Set it as an environment variable before launching
Jupyter, e.g. on the command line:

    export ANTHROPIC_API_KEY="sk-ant-........"
    jupyter notebook Agentic_Claims_Investigator.ipynb

If no key is present, the cell below will say so clearly and the rest of
the notebook will not silently fall back to fake output — every response
shown later is a genuine model call.

In [ ]:
import os
import json
import time
from anthropic import Anthropic

MODEL = "claude-sonnet-4-6"

api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    print(
        "No ANTHROPIC_API_KEY found in the environment.\n\n"
        "This notebook calls the real Claude API, so it needs a key to run.\n"
        "Get one at https://console.anthropic.com, then either:\n"
        '  export ANTHROPIC_API_KEY="sk-ant-..."   (before launching Jupyter)\n'
        "or set it directly in this session with:\n"
        '  os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."\n\n'
        "The remaining cells will raise a clear error rather than fabricate output."
    )
else:
    print("ANTHROPIC_API_KEY found — ready to call the Claude API.")

client = Anthropic(api_key=api_key) if api_key else None


def ask_claude(system, user, max_tokens=500):
    """Single real call to the Claude API. Raises clearly if no key is set."""
    if client is None:
        raise RuntimeError(
            "No ANTHROPIC_API_KEY set — cannot call the Claude API. "
            "See the setup instructions above."
        )
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return response.content[0].text


## 2. The flagged claim under investigation

In a connected pipeline this is exactly the shape produced by Node 1's
`handoff_records` (see `Claims_Anomaly_Detector.ipynb`, section 8): each
record carries `provider_id`, `date`, `billed_amount`, `rolling_mean`,
`z_score`, and `reason_code`. Here we hard-code one representative flagged
claim, taken directly from a real Node 1 run, so this notebook can also run
on its own without re-executing Node 1 first.

In [ ]:
flagged_claim = {
    "provider_id": "PRV-2207",
    "provider_name": "Outpatient Imaging Center",
    "claim_date": "2026-05-18",
    "billed_amount": 253.50,
    "rolling_mean": 415.58,
    "z_score": -9.68,
    "reason_code": "9.7σ below the trailing mean ($416)",
}
flagged_claim


## 3. Simulated evidence sources

A real agentic system would call out to live systems — a claims history
database, a provider-network API, a CPT-to-ICD policy mapping store — to
gather this context itself. To keep this demo runnable without external
infrastructure, the "tool outputs" below are hard-coded strings standing in
for what those lookups would return. The Claude calls that follow only ever
see this simulated evidence; nothing here is fabricated by the model.

In [ ]:
provider_history_lookup = (
    "Provider PRV-2207 (Outpatient Imaging Center) billing history: "
    "median claim over trailing 90 days is $410 (MRI and CT imaging studies). "
    "No prior fraud flags on this provider in the last 24 months. "
    "Claim volume has been stable month over month."
)

policy_snippet_lookup = (
    "Payer policy excerpt, Outpatient Imaging Reimbursement Schedule §3.2: "
    "'Claims billed below the contracted minimum for a given CPT code should be "
    "reviewed for undercoding, incomplete documentation, or incorrect modifier use "
    "before payment is issued, as systematic undercoding can indicate billing "
    "system errors rather than fraud.'"
)

print(provider_history_lookup)
print()
print(policy_snippet_lookup)


## 4. Step 1 — Triage: form an initial hypothesis

The first real Claude call sees only the statistical flag itself (no
evidence yet) and is asked to reason about *plausible* explanations for an
anomaly of this shape — a single step of chain reasoning before any
evidence is introduced. This separates "what does the number mean" from
"what does the supporting evidence say," mirroring how a human investigator
would approach a flagged case.

In [ ]:
triage_system = (
    "You are a payment-integrity triage assistant for a health care claims "
    "review team. You are given only a statistical anomaly flag for a single "
    "claim, with no supporting evidence yet. List 2-3 plausible, distinct "
    "explanations for an anomaly of this shape and direction. Be concise: "
    "a short labeled list, no preamble, no markdown headers."
)

triage_user = f"""Flagged claim:
- Provider: {flagged_claim['provider_id']} ({flagged_claim['provider_name']})
- Claim date: {flagged_claim['claim_date']}
- Billed amount: ${flagged_claim['billed_amount']:.2f}
- Provider's trailing rolling mean: ${flagged_claim['rolling_mean']:.2f}
- Z-score: {flagged_claim['z_score']:.2f} ({flagged_claim['reason_code']})

What are the plausible explanations for this anomaly?"""

triage_hypotheses = ask_claude(triage_system, triage_user, max_tokens=300)
print(triage_hypotheses)


## 5. Step 2 — Evidence review

The second real Claude call now receives the simulated provider history and
policy excerpt, and is asked which of the triage hypotheses the evidence
actually supports or rules out. This is the step where chain reasoning
matters most: the model has to weigh new information against its own prior
hypotheses rather than starting fresh.

In [ ]:
evidence_system = (
    "You are the same payment-integrity triage assistant, now reviewing "
    "supporting evidence for a previously flagged claim. Given the evidence, "
    "state which of the earlier hypotheses the evidence supports, weakens, or "
    "rules out, and why. Be concise and concrete. No markdown headers."
)

evidence_user = f"""Flagged claim and your earlier hypotheses:
{triage_hypotheses}

New evidence gathered:
1. Provider history: {provider_history_lookup}
2. Policy excerpt: {policy_snippet_lookup}

Which hypotheses does this evidence support, weaken, or rule out?"""

evidence_review = ask_claude(evidence_system, evidence_user, max_tokens=350)
print(evidence_review)


## 6. Step 3 — Structured recommendation

The final call asks Claude to synthesize both prior steps into a single,
structured recommendation a human investigator can act on directly: a
disposition (one of three fixed categories), a one-sentence reason, and a
confidence level. Constraining the output to a strict JSON schema is what
makes this usable as a real pipeline component rather than free-form prose
— a downstream system could parse this directly into a queue.

Note the disposition options deliberately exclude any autonomous "deny
payment" or "approve payment" action. The agent's job ends at a
recommendation; a human investigator makes the actual payment decision.

In [ ]:
recommendation_system = (
    "You are the same payment-integrity assistant, now finalizing your "
    "investigation. Respond with ONLY a JSON object (no markdown fences, no "
    "extra text) with exactly these keys: "
    '"disposition" (one of "clear", "needs_human_review", or "escalate_for_audit"), '
    '"confidence" (one of "low", "medium", "high"), '
    '"reason" (one sentence, plain language, suitable for an investigator queue).'
)

recommendation_user = f"""Triage hypotheses:
{triage_hypotheses}

Evidence review:
{evidence_review}

Produce your final structured recommendation now."""

recommendation_raw = ask_claude(recommendation_system, recommendation_user, max_tokens=200)

try:
    recommendation = json.loads(recommendation_raw)
except json.JSONDecodeError:
    # Defensive: if the model wraps the JSON in fences despite instructions, strip them.
    cleaned = recommendation_raw.strip().strip("`").replace("json\n", "", 1)
    recommendation = json.loads(cleaned)

recommendation


## 7. Assemble the investigator-facing case file

Pulling every step together into one record is what a real queue entry
would look like: the original statistical flag, the evidence the agent
consulted, its reasoning trail, and its final structured recommendation —
fully auditable end to end.

In [ ]:
case_file = {
    "claim": flagged_claim,
    "evidence_consulted": [provider_history_lookup, policy_snippet_lookup],
    "agent_triage_hypotheses": triage_hypotheses,
    "agent_evidence_review": evidence_review,
    "agent_recommendation": recommendation,
}

print(json.dumps(case_file, indent=2))


## 8. Scaling up: run the chain over every flagged claim

The same three-step chain can run over an entire batch of flagged claims —
this is the loop a real pipeline would run nightly over everything the
statistical detector surfaced that day. Each claim gets its own independent
triage → evidence → recommendation chain (using the same simulated evidence
sources for simplicity; a production system would look up real evidence per
claim).

In [ ]:
more_flagged_claims = [
    flagged_claim,
    {
        "provider_id": "PRV-4471",
        "provider_name": "Specialty Infusion Center",
        "claim_date": "2026-05-27",
        "billed_amount": 1861.37,
        "rolling_mean": 1208.22,
        "z_score": 4.87,
        "reason_code": "4.9σ above the trailing mean ($1,208)",
    },
    {
        "provider_id": "PRV-3358",
        "provider_name": "Physical Therapy Clinic",
        "claim_date": "2026-05-22",
        "billed_amount": 169.63,
        "rolling_mean": 93.10,
        "z_score": 8.63,
        "reason_code": "8.6σ above the trailing mean ($93)",
    },
]


def investigate_claim(claim, evidence_a, evidence_b):
    """Run the full 3-step agentic chain on one flagged claim."""
    t_user = f"""Flagged claim:
- Provider: {claim['provider_id']} ({claim['provider_name']})
- Claim date: {claim['claim_date']}
- Billed amount: ${claim['billed_amount']:.2f}
- Provider's trailing rolling mean: ${claim['rolling_mean']:.2f}
- Z-score: {claim['z_score']:.2f} ({claim['reason_code']})

What are the plausible explanations for this anomaly?"""
    hyp = ask_claude(triage_system, t_user, max_tokens=250)

    e_user = f"""Earlier hypotheses:
{hyp}

New evidence gathered:
1. Provider history: {evidence_a}
2. Policy excerpt: {evidence_b}

Which hypotheses does this evidence support, weaken, or rule out?"""
    rev = ask_claude(evidence_system, e_user, max_tokens=300)

    r_user = f"""Triage hypotheses:
{hyp}

Evidence review:
{rev}

Produce your final structured recommendation now."""
    raw = ask_claude(recommendation_system, r_user, max_tokens=200)
    try:
        rec = json.loads(raw)
    except json.JSONDecodeError:
        cleaned = raw.strip().strip("`").replace("json\n", "", 1)
        rec = json.loads(cleaned)

    return {"claim": claim, "recommendation": rec}


batch_results = []
for c in more_flagged_claims:
    result = investigate_claim(c, provider_history_lookup, policy_snippet_lookup)
    batch_results.append(result)
    time.sleep(0.5)  # light pacing between calls

for r in batch_results:
    c, rec = r["claim"], r["recommendation"]
    print(f"{c['provider_id']}  |  z={c['z_score']:+.2f}  |  "
          f"{rec['disposition']:<20}  |  confidence={rec['confidence']:<6}  |  {rec['reason']}")


## 9. Conclusion

This notebook demonstrates Node 2 of the Cotiviti Agentic Audit Copilot:
rather than a single static recommendation, the assistant runs a real
multi-step reasoning chain per claim — forming hypotheses, weighing
evidence, and producing a structured, auditable recommendation for a human
investigator. Paired with Node 1 (`Claims_Anomaly_Detector.ipynb`), the two
notebooks together form the complete HITL Adjudication Layer pipeline
proposed in the written report: **Node 1 ranks records by financial risk
and flags what's statistically unusual; Node 2 investigates why, and
generates the auditable narrative summary an analyst reviews before any
payment decision is made.** This is the "Deploy the Cotiviti Agentic Audit
Copilot" strategic option made concrete — pre-assembling the supporting
evidence (longitudinal provider patterns, CPT-to-ICD mapping, policy
language) behind every anomaly flag, while reserving final payment sign-off
strictly for certified human analysts.